# Prapemrosesan Data & Augmentasi Tingkat Lanjut

In [1]:
import os
import cv2
import pandas as pd
import numpy as np
import torch
from torch.utils.data import Dataset
from sklearn.model_selection import StratifiedKFold
import albumentations as A
from albumentations.pytorch import ToTensorV2

# ==========================================
# 1. KONFIGURASI JALUR & HIPERPARAMETER
# ==========================================
BASE_PATH = "/kaggle/input/competitions/data-analytics-competition-dac-find-it-2026"
TRAIN_DIR = os.path.join(BASE_PATH, "train")

IMG_SIZE = 384 
BATCH_SIZE = 8
N_FOLDS = 5 

# ==========================================
# 2. PEMBUATAN DATAFRAME & STRATIFIED K-FOLD
# ==========================================
class_names = sorted(os.listdir(TRAIN_DIR))
label2id = {class_name: i for i, class_name in enumerate(class_names)}
id2label = {i: class_name for class_name, i in label2id.items()}

data = []
for class_name in class_names:
    class_dir = os.path.join(TRAIN_DIR, class_name)
    if os.path.isdir(class_dir):
        for img_name in os.listdir(class_dir):
            data.append({
                "image_path": os.path.join(class_dir, img_name),
                "label": label2id[class_name]
            })

df_all = pd.DataFrame(data)

skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=42)
df_all['fold'] = -1

for fold, (train_idx, valid_idx) in enumerate(skf.split(X=df_all, y=df_all['label'])):
    df_all.loc[valid_idx, 'fold'] = fold

# ==========================================
# 3. PIPELINE AUGMENTASI KELAS GRANDMASTER
# ==========================================
train_transforms = A.Compose([
    A.RandomResizedCrop(size=(IMG_SIZE, IMG_SIZE), scale=(0.8, 1.0), p=1.0),
    A.HorizontalFlip(p=0.5),
    
    # PILAR 1: Simulasi Lensa & Guncangan Lengan (Motion Blur)
    A.OneOf([
        A.ImageCompression(p=1.0), 
        A.GaussianBlur(blur_limit=(3, 7), p=1.0),                      
        A.MotionBlur(blur_limit=5, p=1.0), 
        A.MedianBlur(blur_limit=5, p=1.0),                   
    ], p=0.4),
    
    # PILAR 2: Simulasi Noise Layar Sentuh & Sensor Kamera Murah
    A.OneOf([
        A.ISONoise(p=1.0),
        A.GaussNoise(p=1.0),
        A.MultiplicativeNoise(multiplier=(0.9, 1.1), p=1.0),
    ], p=0.4),
    
    # PILAR 3: Simulasi Pantulan Cahaya & Kontras Lingkungan
    A.OneOf([
        A.RandomBrightnessContrast(brightness_limit=0.2, contrast_limit=0.2, p=1.0),
        A.CLAHE(clip_limit=4.0, tile_grid_size=(8, 8), p=1.0),
        A.HueSaturationValue(hue_shift_limit=15, sat_shift_limit=25, val_shift_limit=25, p=1.0),
        A.RandomToneCurve(p=1.0),
    ], p=0.5),
    
    # PILAR 4: Simulasi Kertas Foto Melengkung / Layar Miring (SENJATA RAHASIA)
    A.OneOf([
        A.GridDistortion(num_steps=5, distort_limit=0.3, p=1.0),
        A.OpticalDistortion(distort_limit=0.05, shift_limit=0.05, p=1.0),
        A.Affine(scale=(0.9, 1.1), translate_percent=(-0.1, 0.1), rotate=(-15, 15), p=1.0),
    ], p=0.4),
    
    # Simulasi oklusi (tertutup jari/benda kecil)
    A.CoarseDropout(max_holes=8, max_height=32, max_width=32, min_holes=1, fill_value=0, p=0.3),
    
    A.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    ToTensorV2()
])

valid_transforms = A.Compose([
    A.Resize(height=IMG_SIZE, width=IMG_SIZE),
    A.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    ToTensorV2()
])

# ==========================================
# 4. KELAS PYTORCH DATASET
# ==========================================
class SpoofingDataset(Dataset):
    def __init__(self, df, transforms=None):
        self.df = df.reset_index(drop=True)
        self.transforms = transforms

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        img_path = self.df.loc[idx, 'image_path']
        image = cv2.imread(img_path)
        
        if image is None:
            image = np.zeros((IMG_SIZE, IMG_SIZE, 3), dtype=np.uint8)
        else:
            image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

        if self.transforms:
            augmented = self.transforms(image=image)
            image = augmented['image']

        label = self.df.loc[idx, 'label']
        return image, torch.tensor(label, dtype=torch.long)

print("Prapemrosesan dan Pembagian K-Fold (dengan Augmentasi Ekstrem) Selesai!")
print(f"Total Seluruh Data : {len(df_all)} gambar")

Prapemrosesan dan Pembagian K-Fold (dengan Augmentasi Ekstrem) Selesai!
Total Seluruh Data : 1652 gambar


/tmp/ipykernel_55/3550670259.py:79: UserWarning: Argument(s) 'shift_limit' are not valid for transform OpticalDistortion
  A.OpticalDistortion(distort_limit=0.05, shift_limit=0.05, p=1.0),
/tmp/ipykernel_55/3550670259.py:84: UserWarning: Argument(s) 'max_holes, max_height, max_width, min_holes, fill_value' are not valid for transform CoarseDropout
  A.CoarseDropout(max_holes=8, max_height=32, max_width=32, min_holes=1, fill_value=0, p=0.3),


# Membangun Arsitektur Swin dan CNN

In [2]:
import torch
import torch.nn as nn
import timm
import gc

# ==========================================
# 1. KONFIGURASI PERANGKAT (GPU / CPU)
# ==========================================
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Menggunakan perangkat: {device}")

# ==========================================
# 2. FUNGSI PEMBUAT MODEL (FACTORY FUNCTIONS)
# ==========================================
NUM_CLASSES = 6
MODEL_SWIN_NAME = 'swin_base_patch4_window12_384.ms_in22k'

# --- REVISI: Mengganti EfficientNet dengan ConvNeXt Base ---
MODEL_CNN_NAME = 'convnext_base.fb_in22k_ft_in1k_384'

# --- PABRIK 1: SWIN TRANSFORMER + ReLU CUSTOM HEAD ---
class SwinWithCustomHead(nn.Module):
    def __init__(self, model_name, num_classes):
        super(SwinWithCustomHead, self).__init__()
        
        # Muat tulang punggung tanpa lapisan akhir bawaan
        self.backbone = timm.create_model(
            model_name, 
            pretrained=True, 
            num_classes=0,       
            drop_rate=0.3,       
            attn_drop_rate=0.2   
        )
        
        num_features = self.backbone.num_features
        
        # Kepala klasifikasi kustom dengan ReLU untuk penalaran non-linear
        self.custom_head = nn.Sequential(
            nn.Linear(num_features, 512),
            nn.BatchNorm1d(512),
            nn.ReLU(),
            nn.Dropout(0.4),
            nn.Linear(512, num_classes)
        )

    def forward(self, x):
        features = self.backbone(x)
        output = self.custom_head(features)
        return output

def create_swin_model():
    model = SwinWithCustomHead(MODEL_SWIN_NAME, NUM_CLASSES)
    
    # TEKNIK GRANDMASTER: Layer Freezing (Membekukan 70% tulang punggung awal)
    # Ini akan mencegah model raksasa overfitting pada dataset kecil
    total_params = len(list(model.backbone.parameters()))
    freeze_limit = int(total_params * 0.70)
    for i, param in enumerate(model.backbone.parameters()):
        if i < freeze_limit:
            param.requires_grad = False
            
    return model.to(device)


# --- PABRIK 2: CONVNEXT BASE ---
def create_cnn_model():
    model = timm.create_model(
        MODEL_CNN_NAME, 
        pretrained=True, 
        num_classes=NUM_CLASSES,
        drop_rate=0.3,       
        drop_path_rate=0.2   
    )
    
    # TEKNIK GRANDMASTER: Layer Freezing untuk CNN
    total_params = len(list(model.parameters()))
    freeze_limit = int(total_params * 0.70)
    for i, param in enumerate(model.parameters()):
        if i < freeze_limit:
            param.requires_grad = False
            
    return model.to(device)


# ==========================================
# 3. UJI COBA FUNGSI (SANITY CHECK)
# ==========================================
print("\nMenguji coba pembuatan kedua model elit...")
temp_swin = create_swin_model()
temp_cnn = create_cnn_model()

dummy_input = torch.randn(4, 3, 384, 384).to(device)

with torch.no_grad():
    dummy_out_swin = temp_swin(dummy_input)
    dummy_out_cnn = temp_cnn(dummy_input)

print("Fungsi pembuat model berhasil diverifikasi dan siap mencetak 10 model!")
print(f"Bentuk input           : {dummy_input.shape}")
print(f"Bentuk output Swin     : {dummy_out_swin.shape}")
print(f"Bentuk output ConvNeXt : {dummy_out_cnn.shape}")

# Bersihkan memori GPU secara total agar siap untuk pelatihan berat
del temp_swin, temp_cnn
torch.cuda.empty_cache()
gc.collect()

Menggunakan perangkat: cuda

Menguji coba pembuatan kedua model elit...


model.safetensors:   0%|          | 0.00/451M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/354M [00:00<?, ?B/s]

Fungsi pembuat model berhasil diverifikasi dan siap mencetak 10 model!
Bentuk input           : torch.Size([4, 3, 384, 384])
Bentuk output Swin     : torch.Size([4, 6])
Bentuk output ConvNeXt : torch.Size([4, 6])


80

# Strategi Pelatihan (Warm-up & Loss)

In [3]:
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.utils.data import DataLoader
from sklearn.metrics import f1_score
from tqdm import tqdm
import numpy as np
import gc

# ==========================================
# 1. ALGORITMA MIXUP & CUTMIX (SINTESIS DATA)
# ==========================================
def rand_bbox(size, lam):
    W = size[2]
    H = size[3]
    cut_rat = np.sqrt(1. - lam)
    cut_w = int(W * cut_rat)
    cut_h = int(H * cut_rat)

    cx = np.random.randint(W)
    cy = np.random.randint(H)

    bbx1 = np.clip(cx - cut_w // 2, 0, W)
    bby1 = np.clip(cy - cut_h // 2, 0, H)
    bbx2 = np.clip(cx + cut_w // 2, 0, W)
    bby2 = np.clip(cy + cut_h // 2, 0, H)

    return bbx1, bby1, bbx2, bby2

def mixup_cutmix(x, y, alpha=1.0):
    if alpha > 0:
        lam = np.random.beta(alpha, alpha)
    else:
        lam = 1

    batch_size = x.size()[0]
    index = torch.randperm(batch_size).to(device)
    
    # 50% MixUp, 50% CutMix
    if np.random.rand() > 0.5:
        # MixUp
        mixed_x = lam * x + (1 - lam) * x[index, :]
    else:
        # CutMix
        bbx1, bby1, bbx2, bby2 = rand_bbox(x.size(), lam)
        mixed_x = x.clone()
        mixed_x[:, :, bbx1:bbx2, bby1:bby2] = x[index, :, bbx1:bbx2, bby1:bby2]
        lam = 1 - ((bbx2 - bbx1) * (bby2 - bby1) / (x.size()[-1] * x.size()[-2]))

    y_a, y_b = y, y[index]
    return mixed_x, y_a, y_b, lam

# ==========================================
# 2. DEFINISI FOCAL LOSS (SUPPORT MIXUP/CUTMIX)
# ==========================================
class FocalLoss(nn.Module):
    def __init__(self, alpha=1, gamma=2, reduction='mean', label_smoothing=0.1):
        super(FocalLoss, self).__init__()
        self.alpha = alpha
        self.gamma = gamma 
        self.reduction = reduction
        self.label_smoothing = label_smoothing

    def forward(self, inputs, targets):
        num_classes = inputs.size(1)
        smooth_target = F.one_hot(targets, num_classes=num_classes).float()
        smooth_target = smooth_target * (1.0 - self.label_smoothing) + (self.label_smoothing / num_classes)
        
        log_pt = F.log_softmax(inputs, dim=1)
        pt = torch.exp(log_pt)
        
        ce_loss = -(smooth_target * log_pt).sum(dim=1)
        F_loss = self.alpha * (1 - pt.gather(1, targets.view(-1, 1)).squeeze(1))**self.gamma * ce_loss
        
        if self.reduction == 'mean':
            return torch.mean(F_loss)
        return F_loss

def criterion_mixup(criterion, pred, y_a, y_b, lam):
    return lam * criterion(pred, y_a) + (1 - lam) * criterion(pred, y_b)

# ==========================================
# 3. FUNGSI PELATIHAN GENERIC
# ==========================================
def train_fold(fold, train_loader, valid_loader, model_func, model_prefix, epochs=30, lr=1e-5, patience=10, accumulation_steps=4):
    print(f"\n{'='*50}")
    print(f"Memulai Pelatihan {model_prefix.upper()}: FOLD {fold}")
    print(f"{'='*50}")
    
    model = model_func()
    criterion = FocalLoss(gamma=2.0, label_smoothing=0.1)
    optimizer = optim.AdamW(model.parameters(), lr=lr, weight_decay=0.05)
    
    # REVISI: Cosine Annealing Scheduler yang lebih lambat agar Backbone tidak kaget
    scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs, eta_min=1e-7)
    scaler = torch.cuda.amp.GradScaler()

    best_val_f1 = 0.0
    patience_counter = 0
    save_path = f"best_{model_prefix}_fold_{fold}.pth"

    for epoch in range(epochs):
        # --- FASE TRAINING ---
        model.train()
        train_loss = 0.0
        optimizer.zero_grad() 
        
        train_bar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{epochs} [Train {model_prefix} Fold {fold}]")
        for batch_idx, (images, labels) in enumerate(train_bar):
            images, labels = images.to(device), labels.to(device)
            
            # Terapkan MixUp / CutMix secara acak (50% probabilitas)
            if np.random.rand() < 0.5:
                images, targets_a, targets_b, lam = mixup_cutmix(images, labels, alpha=1.0)
                use_mixup = True
            else:
                use_mixup = False
                
            with torch.cuda.amp.autocast():
                outputs = model(images) 
                
                # Hitung loss sesuai dengan apakah gambar itu dimanipulasi atau tidak
                if use_mixup:
                    loss = criterion_mixup(criterion, outputs, targets_a, targets_b, lam)
                else:
                    loss = criterion(outputs, labels)
                    
                loss = loss / accumulation_steps 
            
            scaler.scale(loss).backward() 
            
            if ((batch_idx + 1) % accumulation_steps == 0) or (batch_idx + 1 == len(train_loader)):
                scaler.step(optimizer) 
                scaler.update()
                optimizer.zero_grad() 
            
            train_loss += loss.item() * accumulation_steps * images.size(0)
            train_bar.set_postfix(loss=(loss.item() * accumulation_steps))
            
        avg_train_loss = train_loss / len(train_loader.dataset)
        
        # --- FASE VALIDASI ---
        model.eval()
        val_loss = 0.0
        all_preds = []
        all_labels = []
        
        val_bar = tqdm(valid_loader, desc=f"Epoch {epoch+1}/{epochs} [Valid {model_prefix} Fold {fold}]")
        with torch.no_grad(): 
            for images, labels in val_bar:
                images, labels = images.to(device), labels.to(device)
                
                with torch.cuda.amp.autocast():
                    outputs = model(images)
                    loss = criterion(outputs, labels)
                
                val_loss += loss.item() * images.size(0)
                preds = torch.argmax(outputs, dim=1)
                
                all_preds.extend(preds.cpu().numpy())
                all_labels.extend(labels.cpu().numpy())
                
        avg_val_loss = val_loss / len(valid_loader.dataset)
        val_macro_f1 = f1_score(all_labels, all_preds, average='macro')
        
        print(f"\nHasil Epoch {epoch+1} ({model_prefix} Fold {fold}):")
        print(f"Train Loss: {avg_train_loss:.4f} | Val Loss: {avg_val_loss:.4f} | Val Macro F1: {val_macro_f1:.4f}")
        
        scheduler.step()
        
        if val_macro_f1 > best_val_f1:
            best_val_f1 = val_macro_f1
            torch.save(model.state_dict(), save_path)
            print(f"Model membaik! Disimpan ke '{save_path}'")
            patience_counter = 0
        else:
            patience_counter += 1
            print(f"Model tidak membaik. Kesabaran: {patience_counter}/{patience}")
            
        if patience_counter >= patience:
            print(f"\nEarly Stopping dipicu untuk {model_prefix} Fold {fold}!")
            break

    print(f"\nPelatihan {model_prefix} Fold {fold} Selesai! Skor Macro F1 Terbaik: {best_val_f1:.4f}")
    
    del model
    torch.cuda.empty_cache()
    gc.collect()
    
    return best_val_f1

# ==========================================
# 4. EKSEKUSI PELATIHAN 10 MODEL SUPER
# ==========================================
swin_fold_scores = []
cnn_fold_scores = []

for fold in range(N_FOLDS):
    train_df_fold = df_all[df_all['fold'] != fold].reset_index(drop=True)
    valid_df_fold = df_all[df_all['fold'] == fold].reset_index(drop=True)
    
    train_dataset_fold = SpoofingDataset(train_df_fold, transforms=train_transforms)
    valid_dataset_fold = SpoofingDataset(valid_df_fold, transforms=valid_transforms)
    
    train_loader_fold = DataLoader(train_dataset_fold, batch_size=BATCH_SIZE, shuffle=True, num_workers=2, drop_last=True)
    valid_loader_fold = DataLoader(valid_dataset_fold, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)
    
    # --- MELATIH SWIN TRANSFORMER ---
    best_swin_f1 = train_fold(
        fold=fold, 
        train_loader=train_loader_fold, 
        valid_loader=valid_loader_fold, 
        model_func=create_swin_model, 
        model_prefix="swin",
        lr=1e-5
    )
    swin_fold_scores.append(best_swin_f1)
    
    # --- MELATIH CONVNEXT BASE ---
    best_cnn_f1 = train_fold(
        fold=fold, 
        train_loader=train_loader_fold, 
        valid_loader=valid_loader_fold, 
        model_func=create_cnn_model, 
        model_prefix="cnn", # Biarkan namanya tetap 'cnn' agar sinkron dengan kode Tahap 4 & 5
        lr=2e-5 
    )
    cnn_fold_scores.append(best_cnn_f1)

# ==========================================
# 5. REKAPITULASI AKHIR
# ==========================================
print(f"\n{'='*50}")
print("REKAPITULASI PELATIHAN 10-MODEL ENSEMBLE (KAGGLER MODE)")
print(f"{'='*50}")
print("--- SWIN TRANSFORMER (CUSTOM HEAD ReLU) ---")
for i, score in enumerate(swin_fold_scores):
    print(f"Fold {i} F1-Score : {score:.4f}")
print(f"Rata-rata Swin CV F1-Score : {np.mean(swin_fold_scores):.4f}\n")

print("--- CONVNEXT BASE (SPESIALIS TEKSTUR) ---")
for i, score in enumerate(cnn_fold_scores):
    print(f"Fold {i} F1-Score : {score:.4f}")
print(f"Rata-rata ConvNeXt CV F1-Score  : {np.mean(cnn_fold_scores):.4f}")


Memulai Pelatihan SWIN: FOLD 0


/tmp/ipykernel_55/3918369797.py:97: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler()
Epoch 1/30 [Train swin Fold 0]:   0%|          | 0/165 [00:00<?, ?it/s]/tmp/ipykernel_55/3918369797.py:120: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast():
Epoch 1/30 [Valid swin Fold 0]:   0%|          | 0/42 [00:00<?, ?it/s]/tmp/ipykernel_55/3918369797.py:154: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast():
Epoch 1/30 [Valid swin Fold 0]: 100%|██████████| 42/42 [00:08<00:00,  4.91it/s]



Hasil Epoch 1 (swin Fold 0):
Train Loss: 1.3035 | Val Loss: 0.9707 | Val Macro F1: 0.4407
Model membaik! Disimpan ke 'best_swin_fold_0.pth'


Epoch 2/30 [Train swin Fold 0]:   0%|          | 0/165 [00:00<?, ?it/s]/tmp/ipykernel_55/3918369797.py:120: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast():
Epoch 2/30 [Valid swin Fold 0]:   0%|          | 0/42 [00:00<?, ?it/s]/tmp/ipykernel_55/3918369797.py:154: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast():
Epoch 2/30 [Valid swin Fold 0]: 100%|██████████| 42/42 [00:07<00:00,  5.39it/s]



Hasil Epoch 2 (swin Fold 0):
Train Loss: 1.1045 | Val Loss: 0.7640 | Val Macro F1: 0.6044
Model membaik! Disimpan ke 'best_swin_fold_0.pth'


...


Hasil Epoch 23 (cnn Fold 4):
Train Loss: 0.4626 | Val Loss: 0.3306 | Val Macro F1: 0.8311
Model tidak membaik. Kesabaran: 8/10


Epoch 24/30 [Train cnn Fold 4]:   0%|          | 0/165 [00:00<?, ?it/s]/tmp/ipykernel_55/3918369797.py:120: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast():
Epoch 24/30 [Valid cnn Fold 4]:   0%|          | 0/42 [00:00<?, ?it/s]/tmp/ipykernel_55/3918369797.py:154: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast():
Epoch 24/30 [Valid cnn Fold 4]: 100%|██████████| 42/42 [00:06<00:00,  6.72it/s]



Hasil Epoch 24 (cnn Fold 4):
Train Loss: 0.4457 | Val Loss: 0.3281 | Val Macro F1: 0.8249
Model tidak membaik. Kesabaran: 9/10


Epoch 25/30 [Train cnn Fold 4]:   0%|          | 0/165 [00:00<?, ?it/s]/tmp/ipykernel_55/3918369797.py:120: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast():
Epoch 25/30 [Valid cnn Fold 4]:   0%|          | 0/42 [00:00<?, ?it/s]/tmp/ipykernel_55/3918369797.py:154: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast():
Epoch 25/30 [Valid cnn Fold 4]: 100%|██████████| 42/42 [00:06<00:00,  6.78it/s]



Hasil Epoch 25 (cnn Fold 4):
Train Loss: 0.4801 | Val Loss: 0.3307 | Val Macro F1: 0.8244
Model tidak membaik. Kesabaran: 10/10

Early Stopping dipicu untuk cnn Fold 4!

Pelatihan cnn Fold 4 Selesai! Skor Macro F1 Terbaik: 0.8382

REKAPITULASI PELATIHAN 10-MODEL ENSEMBLE (KAGGLER MODE)
--- SWIN TRANSFORMER (CUSTOM HEAD ReLU) ---
Fold 0 F1-Score : 0.8182
Fold 1 F1-Score : 0.8238
Fold 2 F1-Score : 0.8028
Fold 3 F1-Score : 0.8437
Fold 4 F1-Score : 0.8553
Rata-rata Swin CV F1-Score : 0.8287

--- CONVNEXT BASE (SPESIALIS TEKSTUR) ---
Fold 0 F1-Score : 0.7903
Fold 1 F1-Score : 0.8200
Fold 2 F1-Score : 0.7886
Fold 3 F1-Score : 0.8326
Fold 4 F1-Score : 0.8382
Rata-rata ConvNeXt CV F1-Score  : 0.8140


# Kalibrasi Probabilitas (Thresholding)

In [4]:
import torch
import torch.nn.functional as F
import numpy as np
from sklearn.metrics import f1_score, classification_report
from torch.utils.data import DataLoader
import warnings
import gc

# Mengabaikan warning scipy agar output terminal tetap bersih
warnings.filterwarnings("ignore")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
NUM_CLASSES = 6
N_FOLDS = 5 

# Menyiapkan array kosong untuk menampung probabilitas (Swin, ConvNeXt, dan Gabungan)
oof_probs_swin = np.zeros((len(df_all), NUM_CLASSES))
oof_probs_cnn = np.zeros((len(df_all), NUM_CLASSES))
oof_labels = np.zeros(len(df_all))

# ==========================================
# 1. EKSTRAKSI PROBABILITAS OUT-OF-FOLD (OOF)
# ==========================================
print("=== EKSTRAKSI PROBABILITAS OUT-OF-FOLD (10 MODEL) ===")

for fold in range(N_FOLDS):
    print(f"Mengekstrak prediksi dari Fold {fold}...")
    
    val_idx = df_all[df_all['fold'] == fold].index
    valid_df_fold = df_all.iloc[val_idx].reset_index(drop=True)
    
    valid_dataset_fold = SpoofingDataset(valid_df_fold, transforms=valid_transforms)
    valid_loader_fold = DataLoader(valid_dataset_fold, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)
    
    # --- A. EKSTRAKSI SWIN TRANSFORMER ---
    model_swin = create_swin_model()
    model_swin.load_state_dict(torch.load(f'best_swin_fold_{fold}.pth', map_location=device))
    model_swin.eval()
    
    fold_probs_swin = []
    fold_labels = []
    
    with torch.no_grad():
        for images, labels in valid_loader_fold: 
            images = images.to(device)
            out_swin = model_swin(images)
            probs_swin = F.softmax(out_swin, dim=1) 
            fold_probs_swin.extend(probs_swin.cpu().numpy())
            fold_labels.extend(labels.numpy())
            
    oof_probs_swin[val_idx] = np.array(fold_probs_swin)
    oof_labels[val_idx] = np.array(fold_labels)
    
    del model_swin
    torch.cuda.empty_cache()
    gc.collect()

    # --- B. EKSTRAKSI CONVNEXT BASE ---
    model_cnn = create_cnn_model()
    model_cnn.load_state_dict(torch.load(f'best_cnn_fold_{fold}.pth', map_location=device))
    model_cnn.eval()
    
    fold_probs_cnn = []
    
    with torch.no_grad():
        for images, _ in valid_loader_fold: 
            images = images.to(device)
            out_cnn = model_cnn(images)
            probs_cnn = F.softmax(out_cnn, dim=1) 
            fold_probs_cnn.extend(probs_cnn.cpu().numpy())
            
    oof_probs_cnn[val_idx] = np.array(fold_probs_cnn)
    
    del model_cnn
    torch.cuda.empty_cache()
    gc.collect()

# ==========================================
# 2. PENCARIAN RASIO ENSEMBLE TERBAIK (AUTO-BLENDING)
# ==========================================
print("\n--- MENCARI RASIO ENSEMBLE TERBAIK (SWIN vs CONVNEXT) ---")
best_alpha = 0.5
best_blend_f1 = 0.0

# Komputer akan mencoba 81 rasio berbeda (dari 10% Swin hingga 90% Swin)
for alpha in np.linspace(0.1, 0.9, 81): 
    blend_probs = (oof_probs_swin * alpha) + (oof_probs_cnn * (1.0 - alpha))
    blend_preds = np.argmax(blend_probs, axis=1)
    blend_f1 = f1_score(oof_labels, blend_preds, average='macro')
    
    if blend_f1 > best_blend_f1:
        best_blend_f1 = blend_f1
        best_alpha = alpha

print(f"Rasio Terbaik Ditemukan : {best_alpha:.2f} Swin : {1.0 - best_alpha:.2f} ConvNeXt")
print(f"Macro F1-Score Gabungan : {best_blend_f1:.5f}")

# ==========================================
# 3. EVALUASI AKHIR (MURNI TANPA OFFSETS)
# ==========================================
oof_probs_ensemble = (oof_probs_swin * best_alpha) + (oof_probs_cnn * (1.0 - best_alpha))
preds_after = np.argmax(oof_probs_ensemble, axis=1)

print("\n--- EVALUASI AKHIR ---")
print(classification_report(oof_labels, preds_after, target_names=class_names, digits=4))

=== EKSTRAKSI PROBABILITAS OUT-OF-FOLD (10 MODEL) ===
Mengekstrak prediksi dari Fold 0...
Mengekstrak prediksi dari Fold 1...
Mengekstrak prediksi dari Fold 2...
Mengekstrak prediksi dari Fold 3...
Mengekstrak prediksi dari Fold 4...

--- MENCARI RASIO ENSEMBLE TERBAIK (SWIN vs CONVNEXT) ---
Rasio Terbaik Ditemukan : 0.82 Swin : 0.18 ConvNeXt
Macro F1-Score Gabungan : 0.83120

--- EVALUASI AKHIR ---
                precision    recall  f1-score   support

fake_mannequin     0.8458    0.9062    0.8750       224
     fake_mask     0.8333    0.8633    0.8481       278
  fake_printed     0.6742    0.7542    0.7120       118
   fake_screen     0.9319    0.7773    0.8476       229
  fake_unknown     0.8338    0.8462    0.8399       338
    realperson     0.8712    0.8581    0.8646       465

      accuracy                         0.8444      1652
     macro avg     0.8317    0.8342    0.8312      1652
  weighted avg     0.8481    0.8444    0.8449      1652



# Prediksi TTA & Pengumpulan (Submission)

In [5]:
import os
import cv2
import torch
import numpy as np
import pandas as pd
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import albumentations as A
from albumentations.pytorch import ToTensorV2
from tqdm import tqdm
import warnings
import gc

warnings.filterwarnings("ignore")

# ==========================================
# 1. PERSIAPAN JALUR & TEMPLATE SUBMISSION
# ==========================================
TEST_DIR = "/kaggle/input/competitions/data-analytics-competition-dac-find-it-2026/test"
SAMPLE_SUB_PATH = "/kaggle/input/competitions/data-analytics-competition-dac-find-it-2026/samplesubmission.csv"

sub_df = pd.read_csv(SAMPLE_SUB_PATH)

class_names = ['fake_mannequin', 'fake_mask', 'fake_printed', 'fake_screen', 'fake_unknown', 'realperson']
id2label = {i: class_name for i, class_name in enumerate(class_names)}

IMG_SIZE = 384
BATCH_SIZE = 8 
N_FOLDS = 5

test_transforms = A.Compose([
    A.Resize(height=IMG_SIZE, width=IMG_SIZE),
    A.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    ToTensorV2()
])

# ==========================================
# 2. DATASET ANTI-CRASH (DETEKSI OTOMATIS)
# ==========================================
class TestDatasetTTA(Dataset):
    def __init__(self, df, test_dir, transforms):
        self.df = df
        self.test_dir = test_dir
        self.transforms = transforms
        
        self.existing_files = {}
        if os.path.exists(test_dir):
            for f in os.listdir(test_dir):
                name_without_ext = os.path.splitext(f)[0]
                self.existing_files[name_without_ext] = f

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        img_id = str(self.df.iloc[idx]['id'])
        img_id_clean = os.path.splitext(img_id)[0] 
        
        image = None
        
        if img_id_clean in self.existing_files:
            img_path = os.path.join(self.test_dir, self.existing_files[img_id_clean])
            image = cv2.imread(img_path) 
            
        if image is None:
            image = np.zeros((IMG_SIZE, IMG_SIZE, 3), dtype=np.uint8)
        else:
            image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
        
        augmented_orig = self.transforms(image=image)
        img_orig = augmented_orig['image']
        
        image_flipped = cv2.flip(image, 1) 
        augmented_flip = self.transforms(image=image_flipped)
        img_flip = augmented_flip['image']

        return img_id, img_orig, img_flip

test_dataset = TestDatasetTTA(sub_df, TEST_DIR, transforms=test_transforms)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
NUM_CLASSES = 6

# ==========================================
# [REVISI] SUNTIKAN HASIL AUTO-BLENDING DARI TAHAP 4
# ==========================================
# Memastikan komputer menggunakan rasio terbaik yang ditemukan di Tahap 4
if 'best_alpha' not in globals():
    print("[PERINGATAN] Variabel 'best_alpha' tidak ditemukan! Menggunakan rasio default (60:40).")
    best_alpha = 0.60 

print("\n--- PARAMETER PENGGABUNGAN (BLENDING) YANG DIGUNAKAN ---")
print(f"Rasio Swin Transformer : {best_alpha:.2f}")
print(f"Rasio ConvNeXt Base    : {1.0 - best_alpha:.2f}")

# ==========================================
# 3. PREDIKSI 10-MODEL ENSEMBLE DENGAN TTA
# ==========================================
print(f"\nMemulai prediksi {len(sub_df)} data dengan TTA (10-Model Ensemble Mode)...")

final_swin_probs = np.zeros((len(sub_df), NUM_CLASSES))
final_cnn_probs = np.zeros((len(sub_df), NUM_CLASSES))

# --- BAGIAN A: PREDIKSI 5 SWIN TRANSFORMER ---
print("\n=== EKSEKUSI 5 MODEL SWIN TRANSFORMER ===")
for fold in range(N_FOLDS):
    print(f"Memuat & Memprediksi dengan Swin Fold {fold}...")
    model_swin = create_swin_model()
    model_swin.load_state_dict(torch.load(f'best_swin_fold_{fold}.pth', map_location=device))
    model_swin.eval()

    fold_preds_swin = []
    with torch.no_grad():
        for img_ids, imgs_orig, imgs_flip in tqdm(test_loader, desc=f"Swin TTA Fold {fold}"):
            imgs_orig = imgs_orig.to(device)
            imgs_flip = imgs_flip.to(device)

            out_orig = model_swin(imgs_orig)
            prob_orig = F.softmax(out_orig, dim=1)

            out_flip = model_swin(imgs_flip)
            prob_flip = F.softmax(out_flip, dim=1)

            prob_avg = (prob_orig + prob_flip) / 2.0
            fold_preds_swin.extend(prob_avg.cpu().numpy())

    final_swin_probs += (np.array(fold_preds_swin) / N_FOLDS)

    del model_swin
    torch.cuda.empty_cache()
    gc.collect()

# --- BAGIAN B: PREDIKSI 5 CONVNEXT BASE ---
print("\n=== EKSEKUSI 5 MODEL CONVNEXT BASE ===")
for fold in range(N_FOLDS):
    print(f"Memuat & Memprediksi dengan ConvNeXt Fold {fold}...")
    model_cnn = create_cnn_model()
    model_cnn.load_state_dict(torch.load(f'best_cnn_fold_{fold}.pth', map_location=device))
    model_cnn.eval()

    fold_preds_cnn = []
    with torch.no_grad():
        for img_ids, imgs_orig, imgs_flip in tqdm(test_loader, desc=f"CNN TTA Fold {fold}"):
            imgs_orig = imgs_orig.to(device)
            imgs_flip = imgs_flip.to(device)

            out_orig = model_cnn(imgs_orig)
            prob_orig = F.softmax(out_orig, dim=1)

            out_flip = model_cnn(imgs_flip)
            prob_flip = F.softmax(out_flip, dim=1)

            prob_avg = (prob_orig + prob_flip) / 2.0
            fold_preds_cnn.extend(prob_avg.cpu().numpy())

    final_cnn_probs += (np.array(fold_preds_cnn) / N_FOLDS)

    del model_cnn
    torch.cuda.empty_cache()
    gc.collect()

# ==========================================
# 4. PENERAPAN MURNI & PENYIMPANAN
# ==========================================
print(f"\n--- Menggabungkan 10 Prediksi (Rasio Swin {best_alpha:.2f}) ---")

# Hanya menggunakan rasio Blending (Tanpa Powell Offsets)
final_ensemble_probs = (final_swin_probs * best_alpha) + (final_cnn_probs * (1.0 - best_alpha))
final_preds = np.argmax(final_ensemble_probs, axis=1)

# Simpan ke CSV
final_labels = [id2label[pred] for pred in final_preds]
sub_df['label'] = final_labels
sub_df.to_csv('submission.csv', index=False)

print("Selesai! File 'submission.csv' Super 10-Model (Clean Version) berhasil dibuat.")


--- PARAMETER PENGGABUNGAN (BLENDING) YANG DIGUNAKAN ---
Rasio Swin Transformer : 0.82
Rasio ConvNeXt Base    : 0.18

Memulai prediksi 404 data dengan TTA (10-Model Ensemble Mode)...

=== EKSEKUSI 5 MODEL SWIN TRANSFORMER ===
Memuat & Memprediksi dengan Swin Fold 0...


Swin TTA Fold 0: 100%|██████████| 51/51 [00:39<00:00,  1.29it/s]


Memuat & Memprediksi dengan Swin Fold 1...


Swin TTA Fold 1: 100%|██████████| 51/51 [00:39<00:00,  1.30it/s]


Memuat & Memprediksi dengan Swin Fold 2...


...

Memuat & Memprediksi dengan ConvNeXt Fold 3...


CNN TTA Fold 3: 100%|██████████| 51/51 [00:32<00:00,  1.57it/s]


Memuat & Memprediksi dengan ConvNeXt Fold 4...


CNN TTA Fold 4: 100%|██████████| 51/51 [00:32<00:00,  1.57it/s]



--- Menggabungkan 10 Prediksi (Rasio Swin 0.82) ---
Selesai! File 'submission.csv' Super 10-Model (Clean Version) berhasil dibuat.


In [6]:
# 0,75